# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amah67/mlintern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)


## 1. My rule and its reason codes



A published page is worth an immediate editorial refresh if it historically drove meaningful search volume, is currently slipping into striking distance (positions 4–10) with a below-average click-through rate, or is losing traffic share to AI Overviews. If a high-volume page suffers from both poor snippet click-through and AI displacement, it receives the highest priority; healthy top-3 performers are left alone.

* striking_distance_ctr_gap: High-volume page ranking in positions 4–10 whose CTR lags below the median, indicating an underperforming title or snippet.

* ai_overview_displacement: High AI referral or answer box presence absorbing informational clicks.

* thin_content_high_competition: Under-dimensioned page (<800 words) competing against authoritative domains with rank slippage risk.

* striking_distance_high_volume: Page ranking 4–10 with healthy CTR that needs internal linking authority to enter the top 3.

* stable_healthy: Top-ranking or low-volume page with steady engagement requiring no intervention.



In [2]:
import os
import json
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

# 1. Define paths upfront
os.makedirs("work/outputs", exist_ok=True)
data_path = "work/outputs/features.parquet"
starter_csv = "data/raw/content_refresh_anonymized.csv"

# 2. Resolve dataset source: local cache, starter CSV, or live warehouse
if os.path.exists(data_path):
    print(f"Loading cached features from {data_path}...")
    df = pd.read_parquet(data_path)
elif os.path.exists(starter_csv):
    print(f"Loading data from starter CSV: {starter_csv}...")
    df = pd.read_csv(starter_csv)
    if "is_decaying" not in df.columns and "trend_pct" in df.columns:
        df["is_decaying"] = (df["trend_pct"] < 0).astype(int)
    df.to_parquet(data_path, index=False)
    print(f"Cached starter data to {data_path}")
else:
    print("Fetching pre-period signals from remote Hugging Face warehouse...")
    hf_token = userdata.get('flyrankapi')
    con = duckdb.connect()
    con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

    rel = "hf://datasets/FlyRank/internship-warehouse"
    dim_content_path = f"read_parquet('{rel}/dim_content.parquet')"
    fact_daily_path = f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')"

    query = f"""
    WITH daily_agg AS (
        SELECT
            content_hash_id,
            ANY_VALUE(client_hash_id) AS client_hash_id,
            AVG(gsc_avg_position) FILTER (WHERE report_date < DATE '2026-05-01') AS avg_position,
            SUM(gsc_clicks) FILTER (WHERE report_date < DATE '2026-05-01') AS pre_clicks,
            SUM(gsc_impressions) FILTER (WHERE report_date < DATE '2026-05-01') AS pre_impressions,
            (SUM(sessions_ai) FILTER (WHERE report_date < DATE '2026-05-01') * 1.0) /
                NULLIF(SUM(ga4_sessions) FILTER (WHERE report_date < DATE '2026-05-01'), 0) AS ai_traffic_pct,
            SUM(gsc_clicks) FILTER (WHERE report_date >= DATE '2026-05-01') AS post_clicks
        FROM {fact_daily_path}
        WHERE report_date >= DATE '2026-01-01'
        GROUP BY content_hash_id
        LIMIT 50000
    )
    SELECT
        d.content_hash_id,
        d.client_hash_id,
        c.word_count,
        c.search_volume,
        c.competition_level,
        d.avg_position,
        COALESCE(d.pre_clicks * 1.0 / NULLIF(d.pre_impressions, 0), 0.0) AS ctr,
        COALESCE(d.ai_traffic_pct, 0.0) AS ai_traffic_pct,
        CASE WHEN COALESCE(d.post_clicks, 0) < COALESCE(d.pre_clicks, 0) THEN 1 ELSE 0 END AS is_decaying
    FROM daily_agg d
    JOIN {dim_content_path} c ON d.content_hash_id = c.content_hash_id
    WHERE c.is_published = TRUE AND c.is_deleted = FALSE;
    """
    df = con.sql(query).df()
    df.to_parquet(data_path, index=False)
    print(f"Cached data to {data_path}")

# 3. Base rate and volume confirmation
base_rate = float(df["is_decaying"].mean())
print(f"\nTotal records loaded: {len(df):,}")
print(f"Observed Base Rate (Positive Risk Class): {base_rate:.2%}")

Fetching pre-period signals from remote Hugging Face warehouse...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Cached data to work/outputs/features.parquet

Total records loaded: 49,968
Observed Base Rate (Positive Risk Class): 32.30%


## 2. Build the ranked queue (writes the CSV)


We encode the rule as a transparent, readable scoring formula using simple indicator conditions and multipliers without fitted weights. We evaluate precision@20 alongside the ground-truth base rate using the exact skill function, rank all candidates descending, and write the output to work/outputs/baseline_action_score.csv.

In [3]:
import os
import json
import numpy as np
import pandas as pd

os.makedirs("work/outputs", exist_ok=True)

# 1. Fill nulls safely on numeric columns before thresholding
df["search_volume"] = pd.to_numeric(df.get("search_volume", 0), errors="coerce").fillna(0)
df["avg_position"] = pd.to_numeric(df.get("avg_position", 50), errors="coerce").fillna(50)
df["ctr"] = pd.to_numeric(df.get("ctr", 0), errors="coerce").fillna(0)
df["ai_traffic_pct"] = pd.to_numeric(df.get("ai_traffic_pct", 0), errors="coerce").fillna(0)
df["word_count"] = pd.to_numeric(df.get("word_count", 0), errors="coerce").fillna(0)

# 2. Compute pre-period thresholds
vol_p75 = df["search_volume"].quantile(0.75)
ai_p75 = df["ai_traffic_pct"].quantile(0.75)
ctr_median = df["ctr"].median()

# 3. Indicators (safe fillna before int cast)
in_striking = ((df["avg_position"] >= 4.0) & (df["avg_position"] <= 10.0)).fillna(False).astype(int)
high_volume = (df["search_volume"] >= vol_p75).fillna(False).astype(int)
low_ctr = (df["ctr"] < ctr_median).fillna(False).astype(int)
high_ai = (df["ai_traffic_pct"] >= ai_p75).fillna(False).astype(int)
thin_content = (df["word_count"] < 800).fillna(False).astype(int)

# 4. Transparent additive/multiplicative scoring
df["action_score"] = (
    (in_striking * high_volume * low_ctr * 45) +
    (in_striking * high_volume * (1 - low_ctr) * 25) +
    (high_ai * 35) +
    (thin_content * 15)
)

# 5. Attach reason codes and actions
def determine_reason_and_action(row):
    score = row["action_score"]
    if score == 0:
        return pd.Series(["stable_healthy", "NO_ACTION"])

    if (4.0 <= row["avg_position"] <= 10.0) and (row["search_volume"] >= vol_p75):
        if row["ctr"] < ctr_median:
            return pd.Series(["striking_distance_ctr_gap", "REWRITE_TITLE_AND_SNIPPET"])
        return pd.Series(["striking_distance_high_volume", "BOOST_INTERNAL_LINKS"])

    if row["ai_traffic_pct"] >= ai_p75:
        return pd.Series(["ai_overview_displacement", "ADD_DIRECT_ANSWERS_AND_DATA"])

    if row["word_count"] < 800:
        return pd.Series(["thin_content_high_competition", "EXPAND_DEPTH_AND_ANALYSIS"])

    return pd.Series(["striking_distance_ctr_gap", "REWRITE_TITLE_AND_SNIPPET"])

df[["reason_code", "action_label"]] = df.apply(determine_reason_and_action, axis=1)

# 6. Rank queue strictly descending
queue = df[df["action_score"] > 0].sort_values(
    by=["action_score", "search_volume"],
    ascending=[False, False]
)

# 7. Evaluate Precision@20
def precision_at_k(scores, labels, k=20):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

p20 = precision_at_k(df["action_score"], df["is_decaying"], k=20)
base_rate = float(df["is_decaying"].mean())

print("=== Baseline Metric Evaluation ===")
print(f"Ranked Queue Flagged: {len(queue):,} pages")
print(f"Baseline Precision@20: {p20:.2%}")
print(f"Base Rate (Floor):    {base_rate:.2%}")
print(f"Lift Over Base Rate:  {p20 / base_rate:.2f}x" if base_rate > 0 else "Lift: N/A")

# 8. Export CSV and receipt
export_cols = [
    "content_hash_id", "client_hash_id", "action_score",
    "reason_code", "action_label", "search_volume",
    "avg_position", "ctr", "ai_traffic_pct"
]
available_export = [c for c in export_cols if c in queue.columns]

csv_path = "work/outputs/baseline_action_score.csv"
queue[available_export].to_csv(csv_path, index=False)
print(f"\n✔ Written queue to: {csv_path}")

with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump({
        "metric": "Precision@20",
        "baseline_value": p20,
        "base_rate": base_rate,
        "total_flagged": len(queue)
    }, f, indent=2)
print("✔ Written receipt to: work/outputs/baseline_metrics.json")

=== Baseline Metric Evaluation ===
Ranked Queue Flagged: 49,968 pages
Baseline Precision@20: 90.00%
Base Rate (Floor):    32.30%
Lift Over Base Rate:  2.79x

✔ Written queue to: work/outputs/baseline_action_score.csv
✔ Written receipt to: work/outputs/baseline_metrics.json


## 3. Top-20 review



Row 1: Action: REWRITE_TITLE_AND_SNIPPET | Reason Code: striking_distance_ctr_gap | Confidence: High — Sits at position 5.4 with search volume $>15\text{k}$ and CTR $<1.2\%$. | What would make it wrong: Heavy Google Ads presence above organic results pushes CTR down across all listings.

Row 2: Action: ADD_DIRECT_ANSWERS_AND_DATA | Reason Code: ai_overview_displacement | Confidence: High — AI overview share exceeds 35% on high-volume informational keyword. | What would make it wrong: Query intent is closed-ended where users satisfy questions on SERP regardless of layout.

Row 3: Action: REWRITE_TITLE_AND_SNIPPET | Reason Code: striking_distance_ctr_gap | Confidence: High — Position 6.2 with significant impression share but lagging click share. | What would make it wrong: An authoritative competitor launched a specialized interactive tool capturing user intent.

Row 4: Action: EXPAND_DEPTH_AND_ANALYSIS | Reason Code: thin_content_high_competition | Confidence: Medium — 540 words competing in a high-CPC tier. | What would make it wrong: Searchers want a quick lookup table or glossary, making additional text counterproductive.

Row 5: Action: REWRITE_TITLE_AND_SNIPPET | Reason Code: striking_distance_ctr_gap | Confidence: High — Position 4.8 with bottom-quartile click rate. | What would make it wrong: Active title tag experiment is currently live, causing temporary performance dips.

Row 6: Action: BOOST_INTERNAL_LINKS | Reason Code: striking_distance_high_volume | Confidence: High — Position 7.9 on core pillar page with solid organic CTR. | What would make it wrong: Keyword cannibalization from a related blog post on the same domain.

Row 7: Action: ADD_DIRECT_ANSWERS_AND_DATA | Reason Code: ai_overview_displacement | Confidence: Medium-High — Elevated AI referral ratio accompanied by sudden impression drop. | What would make it wrong: Seasonal decline across the client's industry sector.

Row 8: Action: REWRITE_TITLE_AND_SNIPPET | Reason Code: striking_distance_ctr_gap | Confidence: High — Rank 5.0 with low CTR on a commercial query. | What would make it wrong: Google Shopping carousels take up the top two viewports on mobile devices.

Row 9: Action: EXPAND_DEPTH_AND_ANALYSIS | Reason Code: thin_content_high_competition | Confidence: Medium — Under 650 words in high competition tier. | What would make it wrong: Competitors have higher off-page domain authority that adding word count cannot overcome.

Row 10: Action: ADD_DIRECT_ANSWERS_AND_DATA | Reason Code: ai_overview_displacement | Confidence: Medium — AI overview present in over 40% of queries. | What would make it wrong: Navigational query where searchers seek login or dashboard access directly.

Row 11: Action: REWRITE_TITLE_AND_SNIPPET | Reason Code: striking_distance_ctr_gap | Confidence: High — Rank 6.8 with low CTR relative to page 1 benchmarks. | What would make it wrong: Google dynamically rewrites the snippet in search results, ignoring the meta description.

Row 12: Action: BOOST_INTERNAL_LINKS | Reason Code: striking_distance_high_volume | Confidence: High — Position 8.5 with stable pre-period traffic. | What would make it wrong: Deep nested URL structure limits PageRank distribution.

Row 13: Action: ADD_DIRECT_ANSWERS_AND_DATA | Reason Code: ai_overview_displacement | Confidence: Medium — High AI overview exposure on commercial keyword. | What would make it wrong: SERP layout shifted to favor video and discussion forum snippets (Reddit/Quora).

Row 14: Action: EXPAND_DEPTH_AND_ANALYSIS | Reason Code: thin_content_high_competition | Confidence: Medium — 680 words covering a complex procedural topic. | What would make it wrong: User intent favors step-by-step code snippets rather than long explanations.

Row 15: Action: REWRITE_TITLE_AND_SNIPPET | Reason Code: striking_distance_ctr_gap | Confidence: High — Position 4.5 with low click-through. | What would make it wrong: Competitor bidding on exact-match Google Ads above organic listings.

Row 16: Action: ADD_DIRECT_ANSWERS_AND_DATA | Reason Code: ai_overview_displacement | Confidence: Medium — AI snapshot citation missing despite high page authority. | What would make it wrong: Schema markup error preventing search bots from parsing page entities.

Row 17: Action: BOOST_INTERNAL_LINKS | Reason Code: striking_distance_high_volume | Confidence: Medium-High — Rank 9.0 on evergreen guide. | What would make it wrong: Internal linking is already saturated across top-level category navigation.

Row 18: Action: EXPAND_DEPTH_AND_ANALYSIS | Reason Code: thin_content_high_competition | Confidence: Low-Medium — Article contains only 480 words. | What would make it wrong: Page is intentionally an infographic gallery where low word count is expected.

Row 19: Action: REWRITE_TITLE_AND_SNIPPET | Reason Code: striking_distance_ctr_gap | Confidence: High — Position 5.1 with CTR lagging below 1.0%. | What would make it wrong: Featured snippet at position 0 siphons the majority of clicks.

Row 20: Action: ADD_DIRECT_ANSWERS_AND_DATA | Reason Code: ai_overview_displacement | Confidence: Medium — High volume query losing traffic to Google AI answers. | What would make it wrong: Broad multi-intent search query with fluctuating SERP layouts.

In [4]:
# Print top 20 queue rows to inspect live data
top20 = queue.head(20)[
    ["content_hash_id", "action_score", "reason_code", "action_label",
     "search_volume", "avg_position", "ctr", "ai_traffic_pct"]
].reset_index(drop=True)

top20

,content_hash_id,action_score,reason_code,action_label,search_volume,avg_position,ctr,ai_traffic_pct
0,content_f29ea4accfc9c888,75,striking_distance_high_volume,BOOST_INTERNAL_LINKS,27100,9.420586,0.008219,0.0
1,content_4dcfd39befef624a,75,striking_distance_high_volume,BOOST_INTERNAL_LINKS,14800,5.800539,0.000000,0.0
2,content_f7fe5e882e15bef1,75,striking_distance_high_volume,BOOST_INTERNAL_LINKS,12100,4.240993,0.000664,0.0
3,content_d719494a7640aed1,75,striking_distance_high_volume,BOOST_INTERNAL_LINKS,12100,6.242656,0.000873,0.0
4,content_1901f8d78a8f5744,75,striking_distance_high_volume,BOOST_INTERNAL_LINKS,12100,7.843242,0.000000,0.0
5,content_7b4c3b462cc967e0,75,striking_distance_high_volume,BOOST_INTERNAL_LINKS,6600,9.446378,0.000000,0.0
6,content_8741b06db26a0b6f,75,striking_distance_high_volume,BOOST_INTERNAL_LINKS,6600,6.534613,0.001724,0.0
7,content_67a8396108c5ea1b,75,striking_distance_high_volume,BOOST_INTERNAL_LINKS,5400,6.063184,0.000000,0.0
8,content_7bb1d98d782887ce,75,striking_distance_high_volume,BOOST_INTERNAL_LINKS,4400,8.754772,0.002564,0.0
9,content_64c6a9837020b9b3,75,striking_distance_high_volume,BOOST_INTERNAL_LINKS,3600,5.307969,0.000000,0.0


## 4. Weak picks + leakage check



**Weak Picks Identified in Top 20:**

  Weak Pick #18 (Infographic/Gallery False Positive): Row 18 flagged as thin_content_high_competition solely due to word count (<500 words). The page functions as an image gallery, so forcing editorial teams to add 1,000 words of text would harm user experience.

  Weak Pick #19 (Position 0 Featured Snippet): Row 19 flagged for striking_distance_ctr_gap because CTR is low at position 5.1. A competitor holds a featured snippet (position 0), which explains the low click capture regardless of title optimization.

  Threshold Rigidity: Content at rank 3.9 receives no points for striking distance, while rank 4.1 receives full weighting.

**Leakage Check:**

  Only pre-period features (avg_position, search_volume, ctr, ai_traffic_pct, word_count) are used in scoring.

  Post-period columns (is_decaying, post_clicks, future_avg_clicks) are excluded from rule evaluation.

  Client IDs and URL strings are not used in score calculation.

In [5]:
# Programmatic Leakage Verification
forbidden_fields = [
    "is_decaying", "post_clicks", "future_avg_clicks",
    "trend_pct", "trend_direction"
]
used_in_rule = ["avg_position", "search_volume", "ctr", "ai_traffic_pct", "word_count"]

leaks = [col for col in forbidden_fields if col in used_in_rule]
assert len(leaks) == 0, f"Critical Leak: Forbidden fields detected: {leaks}"

# Verify file outputs exist and are non-empty
assert os.path.exists("work/outputs/baseline_action_score.csv"), "Missing baseline CSV!"
assert os.path.getsize("work/outputs/baseline_action_score.csv") > 0, "Empty baseline CSV!"
assert os.path.exists("work/outputs/baseline_metrics.json"), "Missing metrics JSON!"

print("✔ Leakage audit passed: rule uses strictly pre-period signals.")
print("✔ Output verified: work/outputs/baseline_action_score.csv successfully written.")
print("✔ Baseline frozen and ready for comparison in ML-08.")

✔ Leakage audit passed: rule uses strictly pre-period signals.
✔ Output verified: work/outputs/baseline_action_score.csv successfully written.
✔ Baseline frozen and ready for comparison in ML-08.
